# ATML PA3 · Task 2 — SigLIP–Qwen VLM (Kaggle GPU T4, or Colab T4)

One run through the three curriculum stages, then the final four-condition evaluation.
Jobs: `features` (download ~11k selected COCO images, cache SigLIP features, ~30 min) →
`train` (4,500 updates, resumable every 250 updates) → `evaluate` → `summarize`.
Data selection (IDs, pairs, mismatched mapping) is already committed in `results/task2/data/`.
Re-run until it prints `REMAINING JOBS: none`. Run `MODE = "smoke"` first (~30 min).

**Secrets** (Kaggle: *Add-ons → Secrets*; Colab: key icon): `HF_TOKEN` (write access; used for the
private checkpoint store and model downloads) and `GITHUB_TOKEN` (read access to the private repo;
not needed once the repo is public).

In [ ]:
# --- Run configuration ---
REPO = "https://github.com/rana-taqveem/atml_internal.git"
BRANCH = "main"
STORE_REPO = ""          # "" -> <your-hf-username>/atml-pa3-store (private, created automatically)
MODE = "smoke"           # "smoke" first, then "full"
HOURS = 11.3
GPUS = "0"               # Task 2 is a single sequential run; one GPU is enough

In [ ]:
# --- Setup: secrets, clone, install (identical in every task notebook) ---
import os, subprocess, sys

def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient          # Kaggle
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata                       # Colab
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

os.environ["HF_TOKEN"] = secret("HF_TOKEN")
if STORE_REPO:
    os.environ["PA3_STORE_REPO"] = STORE_REPO
gh = secret("GITHUB_TOKEN")
url = REPO.replace("https://", f"https://{gh}@") if gh else REPO
# Kaggle: /kaggle/working is capped at 20 GB and saved with the notebook, so the repo and all
# checkpoints live in /tmp; only result zips are written to /kaggle/working.
KAGGLE = os.path.exists("/kaggle/working")
WORK = "/kaggle/working" if KAGGLE else "/content"
CLONE = "/tmp/pa3" if KAGGLE else "/content/pa3"
if not os.path.exists(CLONE):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, CLONE], check=True)
else:
    subprocess.run(["git", "-C", CLONE, "pull", "--ff-only"], check=True)
os.chdir(f"{CLONE}/assignment_03/ATML-PA3")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# --- Run all pending Task 2 jobs ---
args = ["--gpus", GPUS, "--hours", str(HOURS)] + (["--smoke"] if MODE == "smoke" else [])
!python -m task2_vlm.run_all {" ".join(args)}

In [ ]:
# --- Peek ---
import glob
for f in glob.glob("results/task2/summary*/final_metrics.csv"):
    import pandas as pd; display(pd.read_csv(f).round(3))

In [ ]:
ZIP_NAME, RESULTS_DIR = 'task2_results', 'results/task2'
# --- Collect: small results into a zip next to the notebook output (also mirrored in the HF store) ---
import shutil
shutil.make_archive(f"{WORK}/{ZIP_NAME}", "zip", ".", RESULTS_DIR)
if os.path.exists("logs"):
    shutil.make_archive(f"{WORK}/{ZIP_NAME}_logs", "zip", ".", "logs")
print("wrote", f"{WORK}/{ZIP_NAME}.zip")